# F5 — prompted baseline arms (`base_fewshot`, `base_fewshot_constrained`)

Runs un-finetuned `Qwen/Qwen3-1.7B` over the 300 `eval_gold` documents twice and writes two
prediction files for `sxl metrics score` to consume on the laptop.

**Before running:**

1. Settings → Accelerator → **`GPU T4 x2`** (the code uses `cuda:0` only, so latency stays
   comparable across arms — SPEC §2.2).
2. Settings → Internet **on** (the pip install and the model download need it).
3. Add Data → the **`sxl-data`** dataset (`eval_gold.jsonl`, `train.jsonl`, `dev.jsonl`).
4. Set `COMMIT_SHA` in the install cell to the commit you want to measure.

This notebook is thin on purpose: every line of logic lives in `src/sxl/` and is unit-tested
on the laptop. **Stop the session manually when it finishes — idle sessions burn quota.**

In [ ]:
# HF_HOME must be set BEFORE anything imports huggingface_hub: the cache path is
# frozen into module constants at import time. The default lives on /kaggle/working,
# which is only 20 GB and is the persisted notebook output — a 3.4 GB model plus
# safetensors staging would fill it. /kaggle/tmp is ~60 GB of scratch (SPEC §2.2).
import os

os.environ["HF_HOME"] = "/kaggle/tmp/hf"
os.makedirs("/kaggle/tmp/hf", exist_ok=True)

import time

SESSION_STARTED = time.time()  # GPU-hour accounting; printed in the last cell

In [ ]:
# Pin a commit SHA, never a branch (SPEC §2.4): a mid-session push must not change
# what a running notebook is executing.
COMMIT_SHA = "REPLACE_WITH_COMMIT_SHA"
assert COMMIT_SHA != "REPLACE_WITH_COMMIT_SHA", "pin the commit you are measuring"

REPO = "https://github.com/RazaAli1010/schema-extract-lab"

# No flash-attn (needs sm80) and no vLLM (banned, SPEC §5.3 — Turing support is degrading).
!pip install -q "sxl[gpu] @ git+{REPO}@{COMMIT_SHA}"

In [ ]:
# Version banner (SPEC §5.7). Printed into the saved output so a stale Kaggle base
# image is visible in the artifact rather than being a mystery six weeks later.
import importlib.metadata as md

import torch

for pkg in ("torch", "transformers", "trl", "peft", "bitsandbytes", "outlines", "sxl"):
    try:
        print(f"{pkg:>14} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:>14} NOT INSTALLED")

name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"\n{name}  capability={capability}  n_gpus={torch.cuda.device_count()}")

# Fail loudly rather than quietly measuring a P100: every latency and cost number
# downstream is labelled "Tesla T4", and sm_75 is also what forces fp16 + sdpa.
assert capability == (7, 5), f"expected a T4 (7, 5), got {capability} on {name}"

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The package is a pip install here, so `config.ROOT` points into site-packages and
# the default data paths are wrong. Pass all of them explicitly (see config.py).
DATA = "/kaggle/input/sxl-data"
GOLD = f"{DATA}/eval_gold.jsonl"
TRAIN = f"{DATA}/train.jsonl"
OUT = "/kaggle/working/predictions"

os.makedirs(OUT, exist_ok=True)
!ls -la {DATA}
!wc -l {GOLD} {TRAIN}

In [ ]:
# Smoke run first: 8 documents, ~1 minute. Inspect the raw output by eye before
# spending 15 minutes on the full arm.
!sxl gpu predict --arm base_fewshot --limit 8 --gold {GOLD} --train {TRAIN} --out /kaggle/working/_smoke.jsonl

In [ ]:
# What did it actually say? A schema-shaped object with no prose and no <think>
# block is what a healthy run looks like.
import json

with open("/kaggle/working/_smoke.jsonl", encoding="utf-8") as fh:
    smoke = [json.loads(line) for line in fh]

print(f"{sum(r['schema_valid'] for r in smoke)}/{len(smoke)} schema-valid\n")
print(smoke[0]["raw_output"][:1500])

In [ ]:
# Arm 1 of 2: greedy, unconstrained. The competitor that matters (SPEC §3.6).
# Resumable — if the session dies, re-run this cell and it picks up from the
# .partial.jsonl file without regenerating completed documents.
!sxl gpu predict --arm base_fewshot --gold {GOLD} --train {TRAIN} --out {OUT}/base_fewshot.jsonl

In [ ]:
# Arm 2 of 2: Outlines schema-constrained decoding. Slower per token (logit masking)
# and sequential, so budget appreciably longer than arm 1. Expect schema_valid_rate
# ~1.0 and macro_f1 close to arm 1 — that gap is the point of the arm.
!sxl gpu predict --arm base_fewshot_constrained --gold {GOLD} --train {TRAIN} --out {OUT}/base_fewshot_constrained.jsonl

In [ ]:
# Acceptance criteria, asserted in the artifact itself.
ARMS = ("base_fewshot", "base_fewshot_constrained")
gold_ids = [json.loads(line)["doc_id"] for line in open(GOLD, encoding="utf-8")]

for arm in ARMS:
    with open(f"{OUT}/{arm}.jsonl", encoding="utf-8") as fh:
        records = [json.loads(line) for line in fh]

    assert len(records) == 300, (arm, len(records))
    assert [r["doc_id"] for r in records] == gold_ids, f"{arm}: doc_ids drifted from eval_gold"
    assert not any("<think>" in r["raw_output"] for r in records), f"{arm}: thinking mode leaked"

    n_valid = sum(r["schema_valid"] for r in records)
    n_trunc = sum(r["completion_tokens"] >= 512 for r in records)
    print(f"{arm:>26}  valid {n_valid}/300 ({n_valid / 3:.1f}%)  truncated {n_trunc}")

assert (
    sum(json.loads(line)["schema_valid"] for line in open(f"{OUT}/base_fewshot_constrained.jsonl"))
    >= 297
), "constrained arm must reach schema_valid_rate >= 0.99"

print(f"\nelapsed: {(time.time() - SESSION_STARTED) / 3600:.2f} GPU-hours (budget: < 4)")
!ls -la {OUT}

## Back on the laptop

Download `predictions/base_fewshot.jsonl` and `predictions/base_fewshot_constrained.jsonl`
from this notebook's output into `artifacts/predictions/`, then:

```bash
sxl metrics score --arm base_fewshot
sxl metrics score --arm base_fewshot_constrained
sxl metrics compare
```

Whatever the numbers are, they ship (SPEC §1.1). Do not tune the prompt to move them.

**Now stop the session** (Run → Stop session) — idle sessions keep burning the weekly quota.